In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout, SimpleRNN, LSTM, LayerNormalization, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import re
import scipy.io
import psutil
import os
import math
import cmath 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler

#infile_path = 'post_total_data.csv'
infile_path = 'post_dist_bb_1GHZ_csv.csv'
#infile_path = 'post_dist_filt_csv.csv'
df = pd.read_csv(infile_path)

xin_real = df['xI_real']
xin_imag = df['xI_imag']
yout_real = df['yout_real']
yout_imag = df['yout_imag']

#t_ns = np.arange(1,300001)
t_ns = np.arange(1,30001)
# # read real, imaginary parts and calculate amplitude
# xin = xin_real + 1j*xin_imag
# xin_amp = abs(xin)
# #xin_phase = phase(xin)
# xinn = xin/max(xin_amp)
# xinn_amp = abs(xinn)

# yout = yout_real + 1j*yout_imag
# yout_amp = abs(yout)
# # normalize output to peak value
# youtn = yout/max(yout_amp)
# youtn_amp = abs(youtn)
#yout_phase = cmath.phase(y_out)

# # compute rms value for each signal
# xin_rms = np.sqrt(np.mean(np.abs(xin)**2))
# yout_rms = np.sqrt(np.mean(np.abs(yout)**2))
# gainrms_dB = 20*math.log10(yout_rms/xin_rms)
# print(gainrms_dB)

y_arr = np.column_stack((xin_real, xin_imag))
# youtn = np.array(youtn)
# youtn_real = youtn.real
# youtn_imag = youtn.imag
X_arr = np.column_stack((yout_real, yout_imag))

X = pd.DataFrame(X_arr, columns=['yout_real', 'yout_imag'])
y = pd.DataFrame(y_arr, columns=['xin_real', 'xin_imag'])







# X_new = xin.iloc[240000:300001]
# y_new = yout.iloc[240000:300001]
# print("Shape of X_new:", X_new.shape)
# # Keep the first 90,000 rows in both dataframes:
# X = xin.iloc[:240000]
# y = yout.iloc[:240000]


print("Shape of y:", y.shape)


Shape of X_new: (6000, 2)
Shape of y: (24000, 2)


In [2]:
def make_time_delays(X_df: pd.DataFrame, y_df: pd.DataFrame, n_delay: int):
    X_arr = X_df.values            # shape (N, 2)
    y_arr = y_df.values            # shape (N, 2)
    N = len(X_arr)
    
    X_td = []
    y_td = []
    # for each t >= n_delay, stack [X[t-n_delay], ..., X[t]] ? window length = (n_delay+1)*2
    for t in range(n_delay, N):
        window = X_arr[t-n_delay:t+1].ravel()
        X_td.append(window)
        y_td.append(y_arr[t])
    
    return np.vstack(X_td), np.vstack(y_td)

def make_time_delays_array(X_arr, n_delay):
    """
    Given X_arr shape (N, 2), returns windows shape (N-n_delay, (n_delay+1)*2)
    """
    windows = []
    for t in range(n_delay, X_arr.shape[0]):
        window = X_arr[t-n_delay:t+1].ravel()
        windows.append(window)
    return np.vstack(windows)

n_delay = 3
X_td, y_td = make_time_delays(X, y, n_delay)
#X_td, y_td = make_time_delays(X_train_s_df, y_train_s_df, n_delay)

print("Windowed X shape:", X_td.shape)  # (90001 - n_delay, (n_delay+1)*2)
print("Windowed y shape:", y_td.shape)  # (90001 - n_delay, 2)

X_new_td = make_time_delays_array(X_new.values, n_delay)
#X_new_td = make_time_delays_array(X_test_s_df.values, n_delay)
#y_new_td = make_time_delays_array(y_new.values, n_delay)
print("Windowed X_new shape:", X_new_td.shape)

Windowed X shape: (23997, 8)
Windowed y shape: (23997, 2)
Windowed X_new shape: (5997, 8)


In [3]:
#LSTM changes in dimensions 
# 1. Convert arrays to DataFrames
X_td_df = pd.DataFrame(X_td)         # shape (89999, 4)
y_td_df = pd.DataFrame(y_td)         # shape (89999, 2)
X_new_td_df = pd.DataFrame(X_new_td) # shape (30000, 4)
#y_new_td_df = pd.DataFrame(y_new_td) # shape (30000, 4)

print("X sequence")
print(X_td_df.head())
print("Y sequence")
print(y_td_df.head())

X_seq = X_td_df.values.reshape(-1, n_delay+1, 2)      
y_seq = y_td_df.values                 
X_new_seq = X_new_td_df.values.reshape(-1, n_delay+1, 2)  
#y_new_seq = y_new_td_df.values.reshape(-1, n_delay+1, 2)

# (Optional) Print shapes to confirm
print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)
print("X_new_seq shape:", X_new_seq.shape)

X sequence
          0         1         2         3         4         5         6  \
0 -0.696038  0.344660 -0.654714  0.285869 -0.607243  0.196769 -0.537535   
1 -0.654714  0.285869 -0.607243  0.196769 -0.537535  0.111959 -0.455517   
2 -0.607243  0.196769 -0.537535  0.111959 -0.455517  0.056543 -0.400492   
3 -0.537535  0.111959 -0.455517  0.056543 -0.400492 -0.082979 -0.351761   
4 -0.455517  0.056543 -0.400492 -0.082979 -0.351761 -0.292929 -0.220011   

          7  
0  0.111959  
1  0.056543  
2 -0.082979  
3 -0.292929  
4 -0.497226  
Y sequence
          0         1
0  0.487058  0.227196
1  0.322832  0.240043
2  0.143219  0.259109
3 -0.037455  0.279570
4 -0.205068  0.296300
X_seq shape: (23997, 4, 2)
y_seq shape: (23997, 2)
X_new_seq shape: (5997, 4, 2)


In [4]:
#NN model

model = Sequential([
    Input(shape=((n_delay+1), 2)),
    LSTM(64, activation='tanh', return_sequences=True),
    LSTM(64, activation='tanh', return_sequences=True),
    LSTM(64, activation='tanh'),
    Dense(128, activation='relu'),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)
])

# # Build the model
# model = Sequential([
#     Input(shape=(2,)),  # Explicitly defining the input shape
#     Dense(32, activation='relu'),
#     Dense(128, activation='relu'),
#     Dense(64, activation='relu'),
#     Dense(2)  # Output layer with 2 neurons for xI_real and xI_imag
# ])

# model = Sequential([
#     # Same input shape as before
#     Input(shape=((n_delay + 1), 2)),
#     SimpleRNN(32, activation='tanh', return_sequences=True),
#     SimpleRNN(32, activation='tanh'),
#     Dense(128),
#     LeakyReLU(alpha=0.2),
#     Dense(64),
#     LeakyReLU(alpha=0.1),  
#     Dense(32, activation='relu'),
#     Dense(2)
# ])

    
# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
#LSTM New seq here

history = model.fit(X_seq, y_seq, epochs=60, validation_split = 0.1, verbose=1)

#history = model.fit(X_seq, y_seq, epochs=12, validation_split = 0.2, verbose=1)

print("Training complete")




# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")


# Predict using the model
#LSTM new seq here
#y_pred_s = model.predict(X_new_seq)
y_new_pred = model.predict(X_new_seq) 
#y_new_pred = y_scaler.inverse_transform(y_pred_s)
#y_new_pred = model.predict(y_new_seq) 

# Create the DataFrame with predictions
df_predict = pd.DataFrame({
    'xI_predict_real': y_new_pred[:, 0],
    'xI_predict_imag': y_new_pred[:, 1]
})


# Create complex numbers
df_predict['xI_predict'] = (df_predict['xI_predict_real'] + 1j * df_predict['xI_predict_imag']).astype(np.complex128)

# Prepare data for MATLAB file generation
data_to_save = {'xI_predict': df_predict['xI_predict'].values}


output_mat_file_path = 'C:\\Users\\Luke McCubbin\\Box\\NN PA DPD\\Postdistortion_study_UCSD_10_5_2025\\LSTM_TD3_V1GHz.mat'
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f'Predictions saved to {output_mat_file_path}')


Epoch 1/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.8915 - loss: 0.0242 - val_accuracy: 0.9663 - val_loss: 0.0026
Epoch 2/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9715 - loss: 0.0022 - val_accuracy: 0.9767 - val_loss: 0.0017
Epoch 3/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9760 - loss: 0.0019 - val_accuracy: 0.9796 - val_loss: 0.0013
Epoch 4/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9742 - loss: 0.0017 - val_accuracy: 0.9812 - val_loss: 0.0017
Epoch 5/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9786 - loss: 0.0016 - val_accuracy: 0.9800 - val_loss: 0.0014
Epoch 6/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9777 - loss: 0.0017 - val_accuracy: 0.9833 - val_loss: 0.0013
Epoch 7/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9765 - loss: 0.0016 - val_accuracy: 0.9821 - val_loss: 0.0017
Epoch 8/60
675/675 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9792 - loss: 0.0015 - val_accuracy: 0.

In [5]:
# # R^2 metric for regression
# class R2Score(tf.keras.metrics.Metric):
#     def __init__(self, name="r2", **kwargs):
#         super().__init__(name=name, **kwargs)
#         self.sse = self.add_weight(name="sse", initializer="zeros")
#         self.sst = self.add_weight(name="sst", initializer="zeros")

#     def update_state(self, y_true, y_pred, sample_weight=None):
#         y_true = tf.cast(y_true, tf.float32)
#         y_pred = tf.cast(y_pred, tf.float32)
#         err = y_true - y_pred
#         self.sse.assign_add(tf.reduce_sum(tf.square(err)))
#         mean = tf.reduce_mean(y_true, axis=0)
#         self.sst.assign_add(tf.reduce_sum(tf.square(y_true - mean)))

#     def result(self):
#         return 1.0 - (self.sse / (self.sst + 1e-12))

#     def reset_state(self):
#         self.sse.assign(0.0)
#         self.sst.assign(0.0)

# # Optional: simple attention over time (causal-friendly since it's only over given window)
# class TemporalAttention(tf.keras.layers.Layer):
#     def __init__(self, units=64):
#         super().__init__()
#         self.query = Dense(units, use_bias=False)
#         self.key   = Dense(units, use_bias=False)
#         self.value = Dense(units, use_bias=False)
#         self.norm  = LayerNormalization()

#     def call(self, x):  # x: (B, T, H)
#         q = self.query(x)  # (B,T,U)
#         k = self.key(x)    # (B,T,U)
#         v = self.value(x)  # (B,T,U)
#         scores = tf.matmul(q, k, transpose_b=True) / tf.sqrt(tf.cast(tf.shape(k)[-1], tf.float32))  # (B,T,T)
#         weights = tf.nn.softmax(scores, axis=-1)  # (B,T,T)
#         ctx = tf.matmul(weights, v)               # (B,T,U)
#         ctx = self.norm(ctx + x)                  # residual
#         # Pool over time to a vector
#         return tf.reduce_mean(ctx, axis=1)        # (B,U)

# n_hidden = 128  # try 64→128
# reg = l2(1e-5)

# inp = Input(shape=((n_delay+1), 2))
# x = LSTM(n_hidden, activation="tanh", return_sequences=True)(inp)
# x = Dropout(0.05)(x)
# x = LSTM(n_hidden, activation="tanh", return_sequences=True)(x)
# x = Dropout(0.05)(x)

# # Attention-based temporal pooling (beats just taking the last step in many cases)
# x = TemporalAttention(units=n_hidden)(x)

# x = Dense(128, activation="relu", kernel_regularizer=reg)(x)
# x = Dropout(0.5)(x)
# x = Dense(64, activation="relu", kernel_regularizer=reg)(x)
# x = Dense(32, activation="relu", kernel_regularizer=reg)(x)
# out = Dense(2)(x)

# model = tf.keras.Model(inp, out)

# opt = Adam(learning_rate=1e-3)
# model.compile(optimizer=opt, loss=tf.keras.losses.Huber(delta=1.0), metrics=[R2Score(), "mse", "mae"])

# cb = [
#     tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_mse"),
#     tf.keras.callbacks.ReduceLROnPlateau(factor=0.25, patience=5, min_lr=1e-10, monitor="val_mse")
# ]

# history = model.fit(
#     X_seq, y_seq,
#     epochs=100,                # allow callbacks to stop early
#     batch_size=1024,
#     validation_split=0.1,
#     verbose=1,
#     callbacks=cb
# )

# print("Training complete")




# # Function to parse complex numbers, including those with scientific notation from excel csv file
# def parse_complex_number(s):
#     s = s.replace("i", "j")  
#     # Updated regex to handle scientific notation
#     match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
#     if match:
#         real_part = float(match.group(1))
#         imag_part = float(match.group(2))
#         return real_part, imag_part
#     else:
#         raise ValueError(f"Malformed complex number: {s}")


# # Predict using the model
# #LSTM new seq here
# y_new_pred = model.predict(X_new_seq) 

# # Create the DataFrame with predictions
# df_predict = pd.DataFrame({
#     'xI_predict_real': y_new_pred[:, 0],
#     'xI_predict_imag': y_new_pred[:, 1]
# })

# # Create complex numbers
# df_predict['xI_predict'] = (df_predict['xI_predict_real'] + 1j * df_predict['xI_predict_imag']).astype(np.complex128)

# # Prepare data for MATLAB file generation
# data_to_save = {'xI_predict': df_predict['xI_predict'].values}


# output_mat_file_path = 'C:\\Users\\Luke McCubbin\\Box\\NN PA DPD\\Postdistortion_study_UCSD_10_5_2025\\LSTM_TD3_E50.mat'
# scipy.io.savemat(output_mat_file_path, data_to_save)

# print(f'Predictions saved to {output_mat_file_path}')